In [ ]:
# Install the libraries required for the project

!pip -q install spacy pdfplumber python-docx pandas matplotlib ipywidgets

print("All required libraries have been installed successfully!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 100.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 93.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 80.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 101.4 MB/s eta 0:00:00
All required libraries have been installed successfully!


In [ ]:
# Import the required libraries

import os
import re
import io
import sqlite3
import warnings

import pandas as pd
import matplotlib.pyplot as plt
import spacy
import pdfplumber

from docx import Document
from google.colab import files
from IPython.display import display, clear_output
from datetime import datetime

warnings.filterwarnings("ignore")

print("Libraries imported successfully!")

Libraries imported successfully!


In [ ]:
# Download the English language model

!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 97.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [4]:
# Load the spaCy English language model

nlp = spacy.load("en_core_web_sm")

print("spaCy English NLP model loaded successfully!")

spaCy English NLP model loaded successfully!


In [5]:
# Technical skills our system can recognize

SKILL_KEYWORDS = [
    "Python",
    "Java",
    "C",
    "C++",
    "JavaScript",
    "HTML",
    "CSS",
    "SQL",
    "MySQL",
    "PostgreSQL",
    "MongoDB",
    "Excel",
    "Power BI",
    "Tableau",
    "Machine Learning",
    "Deep Learning",
    "Artificial Intelligence",
    "Natural Language Processing",
    "NLP",
    "Data Analysis",
    "Data Visualization",
    "Pandas",
    "NumPy",
    "Matplotlib",
    "Scikit-learn",
    "TensorFlow",
    "PyTorch",
    "Flask",
    "Django",
    "FastAPI",
    "Git",
    "GitHub",
    "Docker",
    "AWS",
    "Azure",
    "Google Cloud",
    "Data Science",
    "Statistics",
    "Data Structures",
    "Algorithms",
    "REST API",
    "Linux"
]

print("Technical skills database created!")
print("Total skill keywords:", len(SKILL_KEYWORDS))

Technical skills database created!
Total skill keywords: 42


In [6]:
def extract_resume_text(file_name, file_bytes):
    """
    Extract text from PDF or DOCX resume files.
    """

    extension = os.path.splitext(file_name)[1].lower()

    # Extract text from PDF files
    if extension == ".pdf":

        with pdfplumber.open(io.BytesIO(file_bytes)) as pdf:

            pages_text = []

            for page in pdf.pages:
                pages_text.append(page.extract_text() or "")

            return "\n".join(pages_text)

    # Extract text from DOCX files
    elif extension == ".docx":

        document = Document(io.BytesIO(file_bytes))

        paragraphs = [
            paragraph.text
            for paragraph in document.paragraphs
            if paragraph.text.strip()
        ]

        table_text = []

        for table in document.tables:
            for row in table.rows:
                table_text.append(
                    " | ".join(cell.text for cell in row.cells)
                )

        return "\n".join(paragraphs + table_text)

    else:
        raise ValueError(
            "Unsupported file format. Upload a PDF or DOCX resume."
        )


print("Resume text extraction function created successfully!")

Resume text extraction function created successfully!


In [7]:
def extract_email(text):

    email_pattern = r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"

    emails = re.findall(email_pattern, text)

    return emails[0] if emails else "Not Found"


print("Email extraction function created successfully!")

Email extraction function created successfully!


In [8]:
def extract_phone(text):

    phone_pattern = (
        r"(?<!\w)(?:\+?\d{1,3}[\s.-]?)?"
        r"(?:\(?\d{2,5}\)?[\s.-]?)?"
        r"\d{3,5}[\s.-]?\d{3,5}(?!\w)"
    )

    matches = re.findall(phone_pattern, text)

    for match in matches:

        digits = re.sub(r"\D", "", match)

        if 10 <= len(digits) <= 13:
            return match.strip()

    return "Not Found"


print("Phone number extraction function created successfully!")

Phone number extraction function created successfully!


In [9]:
def extract_name(text):

    lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]

    # Look for a likely name near the beginning of the resume
    for line in lines[:8]:

        cleaned_line = re.sub(
            r"[^A-Za-z .'-]",
            "",
            line
        ).strip()

        if (
            2 <= len(cleaned_line.split()) <= 4
            and len(cleaned_line) <= 60
            and not re.search(
                r"resume|curriculum vitae|profile|summary|"
                r"objective|developer|engineer|student|contact",
                cleaned_line,
                re.IGNORECASE
            )
        ):

            doc = nlp(cleaned_line)

            if any(
                ent.label_ == "PERSON"
                for ent in doc.ents
            ):
                return cleaned_line

    # Fallback: look for a person entity in the full text
    doc = nlp(text[:10000])

    for ent in doc.ents:

        if ent.label_ == "PERSON":

            candidate = ent.text.strip()

            if 2 <= len(candidate.split()) <= 4:
                return candidate

    return "Not Found"


print("Candidate name extraction function created successfully!")

Candidate name extraction function created successfully!


In [10]:
def extract_skills(text):

    found_skills = []

    text_lower = text.lower()

    for skill in SKILL_KEYWORDS:

        pattern = (
            r"(?<![a-z0-9])"
            + re.escape(skill.lower())
            + r"(?![a-z0-9])"
        )

        if re.search(pattern, text_lower):
            found_skills.append(skill)

    return sorted(set(found_skills))


print("Technical skills extraction function created successfully!")

Technical skills extraction function created successfully!


In [11]:
def extract_education(text):

    education_keywords = [
        "B.Tech",
        "B.E.",
        "Bachelor of Technology",
        "Bachelor of Engineering",
        "B.Sc",
        "BCA",
        "BBA",
        "B.Com",
        "B.A.",
        "M.Tech",
        "M.E.",
        "M.Sc",
        "MCA",
        "MBA",
        "Master of Technology",
        "Master of Science",
        "Master of Computer Applications",
        "Ph.D",
        "Diploma",
        "Intermediate",
        "Higher Secondary",
        "Class XII",
        "Class X",
        "10th",
        "12th"
    ]

    found_education = []

    for line in text.splitlines():

        for keyword in education_keywords:

            if re.search(
                re.escape(keyword),
                line,
                re.IGNORECASE
            ):

                cleaned_line = line.strip()

                if cleaned_line:
                    found_education.append(cleaned_line)

                break

    return list(dict.fromkeys(found_education))


print("Education extraction function created successfully!")

Education extraction function created successfully!


In [12]:
def extract_education(text):

    education_keywords = [
        "B.Tech",
        "B.E.",
        "Bachelor of Technology",
        "Bachelor of Engineering",
        "B.Sc",
        "BCA",
        "BBA",
        "B.Com",
        "B.A.",
        "M.Tech",
        "M.E.",
        "M.Sc",
        "MCA",
        "MBA",
        "Master of Technology",
        "Master of Science",
        "Master of Computer Applications",
        "Ph.D",
        "Diploma",
        "Intermediate",
        "Higher Secondary",
        "Class XII",
        "Class X",
        "10th",
        "12th"
    ]

    found_education = []

    for line in text.splitlines():

        for keyword in education_keywords:

            if re.search(
                re.escape(keyword),
                line,
                re.IGNORECASE
            ):

                cleaned_line = line.strip()

                if cleaned_line:
                    found_education.append(cleaned_line)

                break

    return list(dict.fromkeys(found_education))


print("Education extraction function created successfully!")

Education extraction function created successfully!


In [13]:
# Connect to the SQLite database

conn = sqlite3.connect(
    "resume_parser.db"
)

cursor = conn.cursor()

# Create the candidates table

cursor.execute("""
CREATE TABLE IF NOT EXISTS candidates (

    id INTEGER PRIMARY KEY AUTOINCREMENT,

    candidate_name TEXT,

    email TEXT,

    phone TEXT,

    skills TEXT,

    education TEXT,

    resume_text TEXT,

    file_name TEXT,

    parsed_at TEXT
)
""")

conn.commit()

print("Candidate database created successfully!")

Candidate database created successfully!


In [14]:
def save_candidate(candidate):

    cursor.execute("""
    INSERT INTO candidates (

        candidate_name,
        email,
        phone,
        skills,
        education,
        resume_text,
        file_name,
        parsed_at

    )

    VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    """, (

        candidate["candidate_name"],
        candidate["email"],
        candidate["phone"],
        ", ".join(candidate["skills"]),
        " | ".join(candidate["education"]),
        candidate["resume_text"],
        candidate["file_name"],
        candidate["parsed_at"]

    ))

    conn.commit()

    print(
        "Candidate saved successfully:",
        candidate["candidate_name"]
    )


print("Candidate database storage function created!")

Candidate database storage function created!


In [15]:
def upload_and_process_resumes():

    print("Upload PDF or DOCX resumes")

    uploaded_files = files.upload()

    results = []

    for file_name, file_bytes in uploaded_files.items():

        try:

            candidate = parse_resume(
                file_name,
                file_bytes
            )

            save_candidate(candidate)

            results.append({

                "Candidate Name":
                    candidate["candidate_name"],

                "Email":
                    candidate["email"],

                "Phone":
                    candidate["phone"],

                "Skills":
                    ", ".join(candidate["skills"]),

                "Education":
                    " | ".join(candidate["education"]),

                "Resume File":
                    candidate["file_name"]

            })

        except Exception as error:

            print(
                f"Could not process {file_name}: {error}"
            )

    if results:

        results_df = pd.DataFrame(results)

        print("\nResume Processing Results")

        display(results_df)

        return results_df

    print("No resumes were processed successfully.")

    return pd.DataFrame()


print("Resume upload system created successfully!")

Resume upload system created successfully!


In [16]:
results_df = upload_and_process_resumes()

Upload PDF or DOCX resumes


Saving Mallemputi Bhavana.pdf to Mallemputi Bhavana.pdf
Could not process Mallemputi Bhavana.pdf: name 'parse_resume' is not defined
No resumes were processed successfully.


In [17]:
def analyze_job_match(candidate_skills, job_description):

    required_skills = extract_skills(
        job_description
    )

    candidate_skill_set = {
        skill.lower()
        for skill in candidate_skills
    }

    required_skill_set = {
        skill.lower()
        for skill in required_skills
    }

    matched_skills = sorted(
        candidate_skill_set & required_skill_set
    )

    missing_skills = sorted(
        required_skill_set - candidate_skill_set
    )

    if required_skill_set:

        match_percentage = (
            len(matched_skills)
            / len(required_skill_set)
        ) * 100

    else:

        match_percentage = 0.0

    return {

        "required_skills": required_skills,

        "matched_skills": matched_skills,

        "missing_skills": missing_skills,

        "match_percentage": round(
            match_percentage,
            2
        )

    }


print("Job description matching function created!")

Job description matching function created!


In [18]:
job_description = """
We are looking for a Python Developer.

Required skills:
Python, SQL, Flask, Git, Machine Learning
"""

# Example candidate skills

candidate_skills = [
    "Python",
    "SQL",
    "Flask",
    "Git"
]

match_result = analyze_job_match(
    candidate_skills,
    job_description
)

print("Required Skills:")
print(match_result["required_skills"])

print("\nMatched Skills:")
print(match_result["matched_skills"])

print("\nMissing Skills:")
print(match_result["missing_skills"])

print(
    "\nSkills Match Percentage:",
    str(match_result["match_percentage"]) + "%"
)

Required Skills:
['Flask', 'Git', 'Machine Learning', 'Python', 'SQL']

Matched Skills:
['flask', 'git', 'python', 'sql']

Missing Skills:
['machine learning']

Skills Match Percentage: 80.0%


In [19]:
def search_candidates_by_skill(skill):

    query = """
    SELECT
        id,
        candidate_name,
        email,
        phone,
        skills,
        education,
        file_name
    FROM candidates
    WHERE skills LIKE ?
    """

    cursor.execute(
        query,
        (f"%{skill}%",)
    )

    rows = cursor.fetchall()

    columns = [
        "ID",
        "Candidate Name",
        "Email",
        "Phone",
        "Skills",
        "Education",
        "Resume File"
    ]

    results = pd.DataFrame(
        rows,
        columns=columns
    )

    if results.empty:

        print(
            f"No candidates found for skill: {skill}"
        )

    else:

        display(results)

    return results


print("Candidate search function created!")

Candidate search function created!


In [20]:
def show_skills_distribution():

    cursor.execute(
        "SELECT skills FROM candidates"
    )

    rows = cursor.fetchall()

    skill_counts = {}

    for row in rows:

        skills = row[0]

        if not skills:
            continue

        for skill in skills.split(","):

            skill = skill.strip()

            if skill:

                skill_counts[skill] = (
                    skill_counts.get(skill, 0) + 1
                )

    if not skill_counts:

        print(
            "No candidate skills available for visualization."
        )

        return

    skills_df = pd.DataFrame(
        list(skill_counts.items()),
        columns=["Skill", "Candidate Count"]
    )

    skills_df = skills_df.sort_values(
        "Candidate Count",
        ascending=False
    ).head(10)

    plt.figure(figsize=(10, 5))

    plt.bar(
        skills_df["Skill"],
        skills_df["Candidate Count"]
    )

    plt.title("Top 10 Skills Across Candidates")

    plt.xlabel("Skills")

    plt.ylabel("Number of Candidates")

    plt.xticks(
        rotation=45,
        ha="right"
    )

    plt.tight_layout()

    plt.show()


print("Skills visualization function created!")

Skills visualization function created!


In [21]:
show_skills_distribution()

No candidate skills available for visualization.


In [22]:
def export_candidates_to_csv():

    query = """
    SELECT
        id,
        candidate_name,
        email,
        phone,
        skills,
        education,
        file_name,
        parsed_at
    FROM candidates
    """

    df = pd.read_sql_query(
        query,
        conn
    )

    if df.empty:

        print(
            "No candidate information available to export."
        )

        return

    output_file = "candidate_analysis_results.csv"

    df.to_csv(
        output_file,
        index=False
    )

    print("Candidate information exported successfully!")

    files.download(output_file)


print("Candidate CSV export function created!")

Candidate CSV export function created!


In [23]:
export_candidates_to_csv()

No candidate information available to export.


In [24]:
def show_candidate_summary():

    cursor.execute(
        "SELECT COUNT(*) FROM candidates"
    )

    total_candidates = cursor.fetchone()[0]

    cursor.execute("""
    SELECT COUNT(DISTINCT email)
    FROM candidates
    WHERE email != 'Not Found'
    """)

    candidates_with_email = cursor.fetchone()[0]

    cursor.execute(
        "SELECT skills FROM candidates"
    )

    rows = cursor.fetchall()

    all_skills = []

    for row in rows:

        if row[0]:

            all_skills.extend(
                skill.strip()
                for skill in row[0].split(",")
                if skill.strip()
            )

    print("=" * 45)

    print("RESUME PARSER - CANDIDATE SUMMARY")

    print("=" * 45)

    print(
        "Total Resume Records:",
        total_candidates
    )

    print(
        "Records with Email Addresses:",
        candidates_with_email
    )

    print(
        "Distinct Skills Identified:",
        len(set(all_skills))
    )

    print("=" * 45)


print("Candidate summary function created!")

Candidate summary function created!
